In [2]:
import os
from deepfix_sdk import DeepFixClient

d:\workspace\repos\deepfix\.venv\Lib\site-packages\deepchecks\core\serialization\dataframe\html.py:16: UserWarning:

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



In [3]:
os.environ["DEEPFIX_API_KEY"] = 'ds_sk_425bb71d_s9Bf6sfc4mvBoOnGVsWOwCWZBw3T3PyIqSeKTtFqc2Y' # get your API key at https://deepfix.delcaux.com

In [16]:
project_id = "2790af97-06d9-4f05-9777-6c4f7def53a9"
url = f"http://localhost:8000/api/analysis/analyse/{project_id}"
client = DeepFixClient(api_url=url, timeout=120)

## Classification

In [5]:
from deepfix_sdk.data.datasets import TabularDataset
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

In [6]:
# Load data
X, y = load_breast_cancer(as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
dataset_name = "breast_cancer_classification"

label = "target"
train = X_train.copy()
train[label] = y_train
cat_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

test = X_test.copy()
test[label] = y_test

In [7]:
train_data = TabularDataset(
    dataset=train, dataset_name=dataset_name, label=label, cat_features=cat_features
)
val_data = TabularDataset(
    dataset=test, dataset_name=dataset_name, label=label, cat_features=cat_features
)

In [8]:
#train_data.data.head()

In [9]:
# Fit model
model_name = "HistGradientBoostingClassifier"
clf = HistGradientBoostingClassifier(max_depth=3)
clf = clf.fit(train_data.X, train_data.y)

In [10]:
result = client.get_diagnosis(
    train_data=train_data,
    test_data=val_data,
    model_name=model_name,
    model=clf,
    language="english",
)

deepchecks - WARNING - Could not find built-in feature importance on the model, using permutation feature importance calculation instead
deepchecks - INFO - Calculating permutation feature importance. Expected to finish in 20 seconds


Connecting to: http://localhost:8000/api/analysis/analyse

Output()

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

In [17]:
result = client.diagnose(
    dataset_name=client.get_dataset_name(train_data, val_data),
    model_name=model_name,
    language="english",
)

Connecting to: http://localhost:8000/api/analysis/analyse/2790af97-06d9-4f05-9777-6c4f7def53a9

Output()

✓ Analysis complete!

In [15]:
# Visualize results
result.to_text(verbose=False)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The integrated analysis reveals a solid but improvable breast cancer classification pipeline: the dataset is         │
│ complete and representative but imbalanced with redundant features, the model config is conservative and unfitted    │
│ (lacking reproducibility and checkpoint), and deepchecks confirm multicollinearity, unused features, and minor       │
│ minority-class underperformance. Key recommendations focus on class weighting, feature selection, hyperparameter     │
│ tuning via CV, and saving a reproducible fitted model to enhance accuracy, fairness, and deployability without major │
│ overhauls, given tree model strengths.                                                                               │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  5                                                            
 Severity Distribution           MEDIUM: 2  HIGH: 2  LOW: 1                                   

                                  HIGH Severity Issues (2)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ High multicollinearity (>20 pairs >0.9   │ Apply RFE or PCA to select 10-15 core    │
│     │ correlation, e.g.,                       │ features, then use SHAP for              │
│     │ radius/perimeter/area) and potential     │ interpretability to integrate or drop    │
│     │ redundancy among derived tumor features, │ unused ones; avoid scaling as tree       │
│     │ with 16 high-variance features unused by │ models are invariant.                    │
│     │ the model.                               │ Reduces variance inflation and model     │
│     │ Evidence: Dataset analyzer flags logical │ instability from correlated inputs,      │
│     │ groupings like mean/worst variants;      │ ensuring full signal use; combines       │
│     │ Deepchecks FAILs on correlations and     │ dataset redundancy with deepchecks       │
│     │ WARNs on unused features; no correlation │ insights to streamline for small N       │
│     │ matrix in checkpoint, but params suggest │ without losing biological info.          │
│     │ no built-in handling.                    │                                          │
│ 2   │ No fitted model checkpoint provided,     │ Train and save the fitted model (e.g.,   │
│     │ only unfitted hyperparameters, rendering │ via joblib) with fixed random_state=42,  │
│     │ it unusable for inference;               │ including metadata like feature names,   │
│     │ random_state=None causes                 │ classes, and train/test splits for       │
│     │ non-reproducibility.                     │ reproducibility.                         │
│     │ Evidence: Model checkpoint lacks         │ Enables deployment and consistent        │
│     │ weights, classes_, or n_iter_;           │ evaluation; integrates across artifacts  │
│     │ random_state=null leads to variable      │ by ensuring the config can be applied to │
│     │ outcomes; Dataset and Deepchecks assume  │ the complete, imbalanced dataset with    │
│     │ a trained model exists for their         │ proper seeding to match deepchecks       │
│     │ analyses, highlighting the gap.          │ results.                                 │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (2)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Class imbalance (63% malignant, 37%      │ Set class_weight='balanced' in           │
│     │ benign) is not addressed in model        │ hyperparameters and monitor              │
│     │ configuration, contributing to 7.14%     │ class-specific metrics like recall for   │
│     │ recall degradation for the minority      │ benign cases during CV; consider SMOTE   │
│     │ class between train and test.            │ if weighting insufficient.               │
│     │ Evidence: Dataset analyzer reports       │ Addresses bias toward majority class     │
│     │ target mean ~0.626 in train/test;        │ without data alteration, improving       │
│     │ Deepchecks notes max degradation in      │ clinical utility for rare benign         │
│     │ Recall for class 0; Model checkpoint     │ detections; integrates dataset imbalance │
│     │ shows class_weight=None, assuming        │ with model config for robust handling in │
│     │ balance despite docstring awareness of   │ small, imbalanced datasets.              │
│     │ imbalance options.                       │                                          │
│ 2   │ Conservative hyperparameters             │ Tune max_depth to 6-10 via grid search   │
│     │ (max_depth=3, min_samples_leaf=20) risk  │ with 5-fold CV, add L1/L2 regularization │
│     │ underfitting complex patterns,           │ if needed, and examine high-PPS features │
│     │ compounded by overly predictive features │ for artifacts before full training.      │
│     │ (PPS >0.7 for worst perimeter/concave    │ Balances underfitting from conservative  │
│     │ points/radius).                          │ settings with overfitting from strong    │
│     │ Evidence: Model checkpoint notes shallow │ predictors; leverages small dataset's    │
│     │ depth limits capacity; Deepchecks flags  │ need for CV to optimize for tumor        │
│     │ high PPS suggesting overfitting          │ variability without overcomplicating.    │
│     │ potential; Dataset shows skewed          │                                          │
│     │ distributions and outliers plausible for │                                          │
│     │ tumors but challenging for shallow       │                                          │
│     │ trees.                                   │                                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                   LOW Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Input assumptions                        │ Specify input as pandas DataFrame in     │
│     │ (categorical_features='from_dtype')      │ docs, use 5-fold CV for variance         │
│     │ mismatch potential array-based dataset   │ assessment, and add input                │
│     │ loading, with minor train-test           │ validation/preprocessing to handle       │
│     │ performance gaps and small dataset size  │ non-categorical formats.                 │
│     │ (455 train) increasing variance risk.    │ Prevents runtime errors and ensures      │
│     │ Evidence: Model checkpoint requires      │ robust evaluation on small data; ties    │
│     │ DataFrame for categoricals but dataset   │ dataset completeness with model          │
│     │ likely loads as arrays; Deepchecks shows │ usability for seamless pipeline from     │
│     │ low degradation but WARN on unused       │ load to inference.                       │
│     │ features; Dataset notes representative   │                                          │
│     │ splits but small N limits                │                                          │
│     │ generalization.                          │                                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''